> **교육 목표** 연관규칙으로 "이런 조건이면 이 고장" 규칙을 찾고, 지지도·신뢰도·향상도를 해석합니다.
>
> **핵심 내용** 구간화 → 빈발 항목 → 규칙 생성 → "조건 → 고장" 규칙만 추리기 → 향상도 순 정렬, 인과로 읽으면 안 되는 이유

In [ ]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D2_06 연관규칙: 어떤 운전 조건에서 어떤 고장이 함께 나나

> **오늘 질문:** "이런 조건이면 이 고장" 규칙을 데이터에서 찾을 수 있을까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_parquet("../../data/factory_table.parquet")
# 공정 온도와 공기 온도의 차이 (작으면 열이 잘 안 빠짐)
df["temp_diff"] = df["process_temp"] - df["air_temp"]

## 1. 구간화: 연속값 → 참/거짓 표 ✍️

연관규칙은 '있다/없다'로 계산합니다. 운전 조건을 '높음/낮음'으로 바꾸고, 고장 유형을 붙여 0/1 표(장바구니)를 만듭니다.

| 컬럼 | 기준 |
|---|---|
| `torque_high` | 토크 상위 10% |
| `rpm_low` / `rpm_high` | 회전수 하위 10% / 상위 10% |
| `wear_high` | 공구 마모 200분 초과 |
| `tempdiff_low` | 온도 차 8.6℃ 미만 |
| `TWF` `HDF` `PWF` `OSF` | 고장 유형 (공구 마모·열 방출·전력·과부하) |

> 변수 이름: `basket`

In [ ]:
# 힌트: pd.DataFrame({"이름": 조건, ...}), df["컬럼"].quantile(0.9)

## 2. 빈발 항목 찾기 ✍️

고장은 드물기 때문에 최소 지지도를 0.3%(1만 건 중 30건)로 낮게 잡습니다.

> 변수 이름: `freq`

In [ ]:
# 힌트: apriori(표, min_support=0.003, use_colnames=True)

## 3. 규칙 만들기 ✍️

향상도(lift)가 2 이상인 규칙만 남깁니다.

> 변수 이름: `rules`

In [ ]:
# 힌트: association_rules(빈발항목, metric="lift", min_threshold=2)

## 4. "조건 → 고장" 규칙만 보기 📋

결과(consequents)가 고장 유형 하나뿐이고, 조건(antecedents)에 고장 유형이 없는 규칙만 고릅니다.

In [ ]:
failure_types = {"TWF", "HDF", "PWF", "OSF"}

keep = []
for _, r in rules.iterrows():
    keep.append(r["consequents"] <= failure_types and len(r["consequents"]) == 1
                and not (r["antecedents"] & failure_types))
cond_rules = rules[keep].copy()

# 보기 좋게 글자로 바꾸기
cond_rules["조건"] = cond_rules["antecedents"].apply(", ".join)
cond_rules["고장"] = cond_rules["consequents"].apply(", ".join)

## 5. 향상도 순으로 정렬 ✍️

In [ ]:
# 힌트: 표.sort_values("컬럼", ascending=False)

## 정리 💬

**Q1.** 향상도가 가장 높은 규칙 두 개를 골라 현장 말로 바꿔 보세요. (지지도·신뢰도·향상도를 모두 써서)

> 답:


**Q2.** 이 규칙들을 "토크가 높으면 고장이 난다"는 인과로 받아들여도 될까요? 현장에서 어떻게 활용하겠습니까?

> 답:
